In [1]:
import pandas as pd
import networkx as nx
from gensim.corpora import Dictionary
from gensim.models import LdaModel
from EmailAnalytics import Email


In [7]:
def test_code(ea: Email):
    print("📌 Testing load_data...")
    df = ea.load_data()
    assert isinstance(df, pd.DataFrame)
    assert "text" in df.columns
    assert pd.api.types.is_datetime64_any_dtype(df["date"])
    assert len(df) == 18
    print("  ✔ load_data crea correctamente el DataFrame base.")

    print("\n📌 Testing build_interaction_graph...")
    graph = ea.build_interaction_graph(include_cc=True)
    assert isinstance(graph, nx.DiGraph)
    assert graph.has_edge("support.agent@ecorp.com", "alice.customer@northwind.com")
    assert graph.has_edge("support.agent@ecorp.com", "lead.support@ecorp.com")
    assert graph["support.agent@ecorp.com"]["lead.support@ecorp.com"]["weight"] == 1
    print("  ✔ build_interaction_graph devuelve un grafo dirigido válido.")

    print("\n📌 Testing analyze_sentiment...")
    df = ea.analyze_sentiment()
    for col in ["polarity", "subjectivity", "sentiment_label"]:
        assert col in df.columns
    assert df["polarity"].between(-1, 1).all()
    assert set(df["sentiment_label"]).issubset({"positive", "neutral", "negative"})
    print("  ✔ analyze_sentiment crea las columnas de sentimiento.")

    print("\n📌 Testing train_topic_model...")
    model, dictionary, corpus = ea.train_topic_model(num_topics=3, passes=10, random_state=42)
    assert isinstance(model, LdaModel)
    assert isinstance(dictionary, Dictionary)
    assert model.num_topics == 3
    assert len(corpus) == 18
    print("  ✔ train_topic_model entrena un modelo LDA válido.")

    print("\n📌 Testing assign_topics...")
    df = ea.assign_topics()
    assert "dominant_topic" in df.columns
    assert "topic_keywords" in df.columns
    assert len(df) == 18
    assert df["dominant_topic"].notna().all()
    print("  ✔ assign_topics anota un tema dominante por correo.")

    print("\n📌 Testing get_topic_report...")
    report = ea.get_topic_report(topn_words=5)
    assert isinstance(report, pd.DataFrame)
    for col in ["topic_id", "keywords", "num_emails", "mean_polarity"]:
        assert col in report.columns
    assert len(report) >= 1
    print("  ✔ get_topic_report devuelve un resumen estructurado por tema.")

    print("\n📌 Testing get_emails_by_sender...")
    subset = ea.get_emails_by_sender("hr@ecorp.com")
    assert isinstance(subset, pd.DataFrame)
    assert len(subset) == 2
    assert (subset["sender"] == "hr@ecorp.com").all()
    print("  ✔ get_emails_by_sender filtra correctamente por remitente.")

    print("\n📌 Testing get_emails_by_topic...")
    first_topic = int(ea.df["dominant_topic"].iloc[0])
    subset_topic = ea.get_emails_by_topic(first_topic)
    assert isinstance(subset_topic, pd.DataFrame)
    assert len(subset_topic) >= 1
    assert (subset_topic["dominant_topic"] == first_topic).all()
    print("  ✔ get_emails_by_topic filtra correctamente por tema.")

    print("\n📌 Testing graph_metrics...")
    metrics = ea.graph_metrics()
    assert isinstance(metrics, dict)
    for key in ["num_nodes", "num_edges", "density"]:
        assert key in metrics
    assert metrics["num_nodes"] > 0
    assert metrics["num_edges"] > 0
    print("  ✔ graph_metrics devuelve las métricas principales del grafo.")

    print("\n🎉 ALL TESTS HAVE BEEN EXECUTED SUCCESSFULLY.")

In [10]:
DATA_PATH = "email_dataset.csv"
ea = Email(DATA_PATH)
test_code(ea)


📌 Testing load_data...
  ✔ load_data crea correctamente el DataFrame base.

📌 Testing build_interaction_graph...


AssertionError: 